[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nepslor/B5203E-TSAF/blob/main/W3/forecastability_indicators_2.ipynb)

### Forecastability KPIs — definitions & interpretation
In this notebook we'll characterize some time series from the M4 weekly dataset, containing series with different level of forecastability. Additionally to the a-priori forecastability indexes we introduced during the lecture, we'll see some more. The complete list is the following:
| KPI | Formula | Meaning | More forecastable when… | Range |
|---|---|---|---|---|
| **Variance ratio** | $ \dfrac{\operatorname{Var}(\Delta y_t)}{\operatorname{Var}(y_t)} $, with $ \Delta y_t = y_t - y_{t-1} $ | Roughness vs overall scale (higher ⇒ rougher) | **Lower** | $[0,\infty)$ |
| **Spectral entropy (normalized)** | $ H_{\text{spec}} = -\dfrac{1}{\log K}\sum_{i=1}^{K} p_i \log p_i,\ \ p_i=\dfrac{P_i}{\sum_j P_j} $ | How concentrated the spectrum is (periodicity) | **Lower** | $[0,1]$ |
| **Spectral forecastability** $\,\Omega$ | $ \Omega = 1 - H_{\text{spec}} $ | Complement of spectral entropy | **Higher** | $[0,1]$ |
| **SVD entropy (normalized)** | $ H_{\text{SVD}} = -\dfrac{1}{\log m}\sum_{i=1}^{m} p_i \log p_i,\ \ p_i=\dfrac{\sigma_i}{\sum_j \sigma_j} $ | Rank-like complexity of delay embedding | **Lower** | $[0,1]$ |
| **Permutation entropy** ($m{=}5,\ \tau{=}1$) | $ H_{\text{perm}} = -\dfrac{1}{\log(m!)} \sum_{\pi} p(\pi)\log p(\pi) $ | Ordinal-pattern unpredictability (amp-invariant) | **Lower** | $[0,1]$ |
| **Sample entropy** ($m{=}2,\ r{=}\,0.2\,\sigma$) | $ \mathrm{SampEn}(m,r) = -\ln\!\left(\dfrac{A}{B}\right) $ | Similar patterns remain similar one step ahead | **Lower** | $[0,\infty)$ (often $0$–$3$) |
| **Approximate entropy** ($m{=}2,\ r{=}\,0.2\,\sigma$) | $ \operatorname{ApEn}(m,r) = \Phi_m(r) - \Phi_{m+1}(r) $ | Older entropy (counts self-matches); regularity | **Lower** | $[0,\infty)$ (often $0$–$3$) |
| **Lempel–Ziv complexity** (8-quantile symbols) | $ C_{\text{LZ}}^{\text{norm}} $ (normalized LZ76 parsing count) | Algorithmic randomness of discretized series | **Lower** | $[0,1]$ |
| **Permutation entropy (ordpy)** | same as above, via ordinal distribution | Entropy on the complexity–entropy plane | **Lower** | $[0,1]$ |
| **Statistical complexity (JS)** | $ C_{\text{JS}} $ (Jensen–Shannon–based complexity from ordinal $p$ vs uniform) | Structured-but-nonrandomness; peaks at intermediate order | **Non-monotone** (descriptive) | typically $[0,1]$ |
| **Autocorr. at lag 1** | $ \rho(1) $ | Linear dependence at 1 step | **Higher in magnitude** ($|\rho(1)|$) | $[-1,1]$ |
| **Autocorr. strength (up to 52)** | $ \sum_{k=1}^{52} |\rho(k)| $ | Total linear structure up to yearly (weekly data) | **Higher** | $[0,52]$ |
| **Periodicity index** | $ \dfrac{\max_{f>0} P(f)}{\sum_{f>0} P(f)} $ | Share of power at the dominant non-DC frequency | **Higher** | $(0,1]$ |
| **Skewness** | standardized third central moment | Distributional asymmetry; candidate for variance-stabilising transforms | Descriptive | $(-\infty,\infty)$ |
| **Zero share** | $n^{-1}\sum_t \mathbb{1}(y_t=0)$ | Intermittency / mass at zero | Descriptive | $[0,1]$ |
| **ADF log-p** | $-\log_{10}(p_{ADF})$ | Evidence against a unit root (trend included) | Higher | $[0,\infty)$ |

### Notes.
For composites, invert the “lower-is-better” metrics, use $|\rho(1)|$ for directionless strength, and consider z-scoring before aggregation. Many entropy/complexity measures prefer z-normalized inputs and enough length.


### Preprocessing tips
Z-normalize; handle missing values; for strongly seasonal data, compute metrics on both raw and seasonally adjusted series. For a monotone composite score, invert all “lower-is-better” metrics, use $|\mathrm{ACF}(1)|$, and optionally transform DFA $\alpha$ to $|\alpha-0.5|$ before z-scoring.


In [ ]:

# Install dependencies
%pip -q install pandas numpy antropy ordpy nolds statsmodels scipy scikit-learn joblib tqdm requests umap-learn anywidget ipywidgets plotly


In [ ]:

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from tqdm import tqdm
from joblib import Parallel, delayed

from scipy import signal
from statsmodels.tsa.stattools import acf

import antropy as ant
import ordpy
import nolds

warnings.filterwarnings("ignore")
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# Toggle for speed: set to True to skip heavier metrics if needed
FAST_MODE = False
N_JOBS = -1  # Use all cores
MAX_LAG_ACF = 52  # weekly: capture ~annual seasonality
PERM_ORDER = 5
SAMPEN_M = 2
APEN_M = 2
R_FRACTION = 0.2  # r = 0.2 * std
LZC_BINS = 8
RANDOM_SEED = 1337
np.random.seed(RANDOM_SEED)


## 1) Download four compact time-series panels

The experiment uses four different domains: **M4 Weekly**, **Tourism Monthly**, **NN5 Daily Cash**, and **Solar 10-minute**. Each source is cached locally after its first download. To keep the notebook practical in Colab, we sample 60 series per source and retain at most the latest 360 observations per series.

In [ ]:
import io
import zipfile

DATASET_SPECS = {
    "M4 weekly": {
        "url": "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Weekly-train.csv",
        "cache_name": "M4_weekly_train.csv",
        "kind": "m4_csv",
        "season": 52,
    },
    "Tourism monthly": {
        "url": "https://zenodo.org/records/4656096/files/tourism_monthly_dataset.zip?download=1",
        "cache_name": "tourism_monthly_dataset.zip",
        "kind": "tsf_zip",
        "season": 12,
    },
    "NN5 daily cash": {
        "url": "https://zenodo.org/records/4656110/files/nn5_daily_dataset_with_missing_values.zip?download=1",
        "cache_name": "nn5_daily_dataset.zip",
        "kind": "tsf_zip",
        "season": 7,
    },
    "Solar 10-minute": {
        "url": "https://zenodo.org/records/4656144/files/solar_10_minutes_dataset.zip?download=1",
        "cache_name": "solar_10_minutes_dataset.zip",
        "kind": "tsf_zip",
        "season": 24,  # one daily cycle at 10-minute resolution
    },
}

N_PER_DATASET = 60
MAX_HISTORY = 360
MIN_HISTORY = 300

def cached_download(spec):
    path = DATA_DIR / spec["cache_name"]
    if not path.exists():
        print(f"Downloading {path.name} ...")
        response = requests.get(spec["url"], timeout=180)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


## 2) Load, sample, and trim the datasets

The loader assigns every series a unique ID that includes the dataset name. Missing values are removed before analysis. The retained history is deliberately short: this keeps the metric calculation and rolling backtests manageable while providing at least two seasonal cycles for the most demanding Solar case.

In [ ]:
from typing import List, Tuple

def parse_m4_csv(path: Path) -> Tuple[List[str], List[np.ndarray]]:
    frame = pd.read_csv(path)
    id_col = frame.columns[0]
    ids, values = [], []
    for row in frame.itertuples(index=False):
        x = np.asarray(row[1:], dtype=float)
        x = x[np.isfinite(x)]
        if len(x):
            ids.append(str(row[0]))
            values.append(x)
    return ids, values

def parse_tsf_bytes(raw: bytes) -> Tuple[List[str], List[np.ndarray]]:
    # Tourism uses Windows-1252 metadata; numeric values are unaffected.
    text = raw.decode("utf-8", errors="replace")
    if "�" in text:
        text = raw.decode("cp1252")
    lines = text.splitlines()
    start = next(i for i, line in enumerate(lines)
                 if line.strip().lower() == "@data") + 1
    ids, values = [], []
    for line in lines[start:]:
        if not line.strip():
            continue
        fields = line.split(":")
        sid = fields[0].strip()
        x = np.asarray([np.nan if v.strip() == "?" else float(v)
                        for v in fields[-1].split(",")])
        x = x[np.isfinite(x)]
        if len(x):
            ids.append(sid)
            values.append(x)
    return ids, values

def load_dataset(dataset, spec, rng):
    path = cached_download(spec)
    if spec["kind"] == "m4_csv":
        ids, values = parse_m4_csv(path)
    else:
        with zipfile.ZipFile(path) as archive:
            tsf_name = next(name for name in archive.namelist() if name.endswith(".tsf"))
            ids, values = parse_tsf_bytes(archive.read(tsf_name))

    # Convert Solar 10-minute observations to hourly means.
    if dataset == "Solar 10-minute":
        hourly_values = []
        for x in values:
            x = np.asarray(x, dtype=float)
            n = (len(x) // 6) * 6
            hourly_values.append(np.nanmean(x[:n].reshape(-1, 6), axis=1))
        values = hourly_values

    eligible = [(sid, x[-MAX_HISTORY:]) for sid, x in zip(ids, values)
                if len(x) >= MIN_HISTORY]
    selected = rng.choice(len(eligible), size=min(N_PER_DATASET, len(eligible)), replace=False)
    return [(dataset, f"{dataset}:{eligible[i][0]}", eligible[i][1], spec["season"])
            for i in selected]

rng = np.random.default_rng(RANDOM_SEED)
panel = []
for dataset, spec in DATASET_SPECS.items():
    panel.extend(load_dataset(dataset, spec, rng))

series_datasets = [dataset for dataset, _, _, _ in panel]
series_ids = [sid for _, sid, _, _ in panel]
series_list = [x for _, _, x, _ in panel]
series_seasons = [season for _, _, _, season in panel]

print(pd.Series(series_datasets).value_counts())
print(f"Total series: {len(series_list)}")


In [ ]:
# Preview examples from each domain.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 2, figsize=(13, 10))
for row, dataset in enumerate(DATASET_SPECS):
    idx = [i for i, d in enumerate(series_datasets) if d == dataset][:2]
    for col, i in enumerate(idx):
        axes[row, col].plot(series_list[i], linewidth=0.9)
        axes[row, col].set_title(series_ids[i])
        axes[row, col].set_xlabel("time step")
plt.tight_layout()


The multi-dataset filtering, sampling, and truncation are already complete. Subsequent sections operate on this 240-series panel.


## 3) Metric functions


In [ ]:

from typing import Dict, Union, Optional
from scipy import signal
from scipy.stats import skew
from statsmodels.tsa.stattools import acf, adfuller

def variance_ratio(x: np.ndarray) -> Optional[float]:
    x = np.asarray(x, float)
    if x.size < 3:
        return np.nan
    num = np.nanvar(np.diff(x))
    den = np.nanvar(x)
    return float(num / den) if den > 0 else np.nan


def adf_logp_safe(x: np.ndarray) -> float:
    """Evidence against a unit root; larger values reject non-stationarity."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 30 or np.nanstd(x) == 0:
        return np.nan
    try:
        pvalue = adfuller(x, regression="ct", autolag="AIC")[1]
        return float(-np.log10(max(pvalue, 1e-16)))
    except Exception:
        return np.nan

def spectral_measures(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, float)
    if x.size < 16:
        return dict(spectral_entropy=np.nan, spectral_forecastability=np.nan, periodicity_index=np.nan)
    try:
        Hs = ant.spectral_entropy(x, sf=1.0, method='welch', normalize=True)
    except Exception:
        Hs = np.nan
    Omega = np.nan if np.isnan(Hs) else 1.0 - Hs
    try:
        freqs, psd = signal.periodogram(x, detrend='linear', scaling='density')
        if psd.size > 1:
            peak = np.nanmax(psd[1:])
            total = np.nansum(psd[1:])
            per_idx = (peak / total) if total and total > 0 else np.nan
        else:
            per_idx = np.nan
    except Exception:
        per_idx = np.nan
    return dict(spectral_entropy=Hs, spectral_forecastability=Omega, periodicity_index=per_idx)

def svd_entropy_safe(x: np.ndarray, order: int = 3, delay: int = 1) -> float:
    if x.size < order * delay + 1:
        return np.nan
    try:
        return ant.svd_entropy(x, order=order, delay=delay, normalize=True)
    except Exception:
        return np.nan

def perm_entropy_safe(x: np.ndarray, order: int = 5, delay: int = 1) -> float:
    if x.size < order * delay + 1:
        return np.nan
    try:
        return ant.perm_entropy(x, order=order, delay=delay, normalize=True)
    except Exception:
        return np.nan

def sample_entropy_safe(x: np.ndarray, m: int = 2, r_fraction: float = 0.2) -> float:
    x = np.asarray(x, float)
    if x.size < 20:
        return np.nan
    s = np.nanstd(x)
    if s == 0:
        return np.nan
    for rf in (r_fraction, 0.25, 0.15, 0.3):
        try:
            val = ant.sample_entropy(x, order=m)
            if np.isfinite(val):
                return float(val)
        except Exception as e:
            pass
    return np.nan

def approximate_entropy_safe(x: np.ndarray, m: int = 2, r_fraction: float = 0.2) -> float:
    x = np.asarray(x, float)
    if x.size < 20:
        return np.nan
    s = np.nanstd(x)
    if s == 0:
        return np.nan
    for rf in (r_fraction, 0.25, 0.15, 0.3):
        try:
            val = ant.app_entropy(x, order=m)
            if np.isfinite(val):
                return float(val)
        except Exception:
            pass
    return np.nan

def quantize_series(x: np.ndarray, n_bins: int = 8) -> np.ndarray:
    x = np.asarray(x, float)
    if np.unique(x).size <= 1:
        return np.zeros_like(x, dtype=int)
    try:
        q = pd.qcut(x, q=min(n_bins, max(2, np.unique(x).size)), labels=False, duplicates='drop')
        return q.astype(int).to_numpy()
    except Exception:
        bins = np.linspace(np.nanmin(x), np.nanmax(x), n_bins + 1)
        return np.digitize(x, bins[:-1], right=False).astype(int)

def lzc_safe(x: np.ndarray, n_bins: int = 8) -> float:
    if x.size < 16:
        return np.nan
    try:
        sym = quantize_series(x, n_bins=n_bins)
        return ant.lziv_complexity(sym, normalize=True)
    except Exception:
        return np.nan

def ordpy_complexity_entropy(x: np.ndarray, dx: int = 5, tau: int = 1) -> Dict[str, float]:
    """
    Returns:
      ordpy_perm_entropy: permutation entropy H (normalized)
      ordpy_stat_complexity: Jensen–Shannon statistical complexity C (normalized)
    Notes:
      - Uses dx (embedding dimension along time).
      - Uses taux=tau, tauy=1, dy=1 (1D patterns).
      - Falls back to old API with `tau` if needed.
    """
    x = np.asarray(x, float)
    if x.size < dx * tau + 1:
        return dict(ordpy_perm_entropy=np.nan, ordpy_stat_complexity=np.nan)
    try:
        # Newer ordpy API
        H, C = ordpy.complexity_entropy(x, dx=dx, dy=1, taux=tau, tauy=1, probs=False)
    except Exception as e:
        print(f"ordpy error: {e}")
        # Older ordpy API fallback
        H, C = ordpy.complexity_entropy(x, dx=dx, taux=tau)
    return dict(ordpy_perm_entropy=float(H), ordpy_stat_complexity=float(C))

def acf_measures(x: np.ndarray, max_lag: int = 52) -> Dict[str, float]:
    if x.size < max_lag + 2:
        return dict(acf1=np.nan, acf_mean_abs=np.nan)
    try:
        a = acf(x, nlags=max_lag, fft=True, missing='drop')
        a1 = float(a[1]) if a.size > 1 else np.nan
        asum = float(np.nanmean(np.abs(a[1:])))
        return dict(acf1=a1, acf_mean_abs=asum)
    except Exception:
        return dict(acf1=np.nan, acf_mean_abs=np.nan)

def dfa_alpha_safe(x: np.ndarray) -> float:
    if x.size < 64:
        return np.nan
    try:
        return float(nolds.dfa(x))
    except Exception:
        return np.nan





## 4) Compute metrics for all series


In [ ]:
def compute_metrics_for_series(sid: str, x: np.ndarray) -> Dict[str, Union[str, float]]:
    raw_x = np.asarray(x, float)
    raw_x = raw_x[np.isfinite(raw_x)]
    x = raw_x.copy()
    x = x - np.nanmean(x)
    s = np.nanstd(x)
    if s > 0:
        x = x / s
    out = {
        "series_id": sid,
        "length": int(x.size),
        "var": float(np.nanvar(x)) if x.size > 0 else np.nan,
        "variance_ratio": variance_ratio(x),
        # Added diagnostic features are computed on the original scale.
        "skewness": float(skew(raw_x, bias=False)) if raw_x.size >= 3 else np.nan,
        "zero_share": float(np.mean(raw_x == 0)) if raw_x.size else np.nan,
        "adf_logp": adf_logp_safe(raw_x),
    }
    out.update(spectral_measures(x))
    out["svd_entropy"] = svd_entropy_safe(x, order=3, delay=1)
    out["perm_entropy"] = perm_entropy_safe(x, order=PERM_ORDER, delay=1)
    if FAST_MODE:
        out["sample_entropy"] = np.nan
        out["approx_entropy"] = np.nan
        out["lzc"] = np.nan
        out["ordpy_perm_entropy"] = np.nan
        out["ordpy_stat_complexity"] = np.nan
    else:
        out["sample_entropy"] = sample_entropy_safe(x, m=SAMPEN_M, r_fraction=R_FRACTION)
        out["approx_entropy"] = approximate_entropy_safe(x, m=APEN_M, r_fraction=R_FRACTION)
        out["lzc"] = lzc_safe(x, n_bins=LZC_BINS)
        oc = ordpy_complexity_entropy(x, dx=PERM_ORDER, tau=1)
        out.update(oc)
    out.update(acf_measures(x, max_lag=MAX_LAG_ACF))
    out["dfa_alpha"] = dfa_alpha_safe(x)
    return out

results = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(compute_metrics_for_series)(sid, x)
                                                  for sid, x in tqdm(zip(series_ids, series_list), total=len(series_list)))


df_metrics = pd.DataFrame(results)
df_metrics["dataset"] = series_datasets
df_metrics["seasonal_period"] = series_seasons
df_metrics.head()



## 5) Composite forecastability index


In [ ]:

from sklearn.preprocessing import StandardScaler
import numpy as np

cols_invert = ["spectral_entropy","svd_entropy","perm_entropy","sample_entropy","approx_entropy",
               "ordpy_perm_entropy","lzc","variance_ratio"]
cols_keep = ["spectral_forecastability","periodicity_index","acf1","acf_mean_abs",
             "ordpy_stat_complexity","dfa_alpha"]

for c in cols_invert:
    if c in df_metrics.columns:
        df_metrics[c + "_inv"] = -df_metrics[c]

use_cols = [c + "_inv" for c in cols_invert if c in df_metrics.columns] + [c for c in cols_keep if c in df_metrics.columns]

df_comp = df_metrics[["series_id"] + use_cols].copy()
df_comp = df_comp.replace([np.inf,-np.inf], np.nan)

# some of the indexes don't have the same scale. Let's normalize them
scaler = StandardScaler()
X = scaler.fit_transform(df_comp[use_cols].to_numpy(dtype=float))


comp_score = np.nanmean(X, axis=1)
df_metrics["forecastability_index"] = comp_score
df_metrics["rank"] = df_metrics["forecastability_index"].rank(ascending=False, method="average")

df_metrics.sort_values("forecastability_index", ascending=False).head(10)


In [ ]:
# plot violinplots of metrics using seaborn
import seaborn as sns
melted = df_metrics.melt(id_vars=["series_id"], value_vars=use_cols, var_name="metric", value_name="value")
plt.figure(figsize=(12, 6))
sns.violinplot(data=melted, x="metric", y="value", inner="quartile", scale="width")
plt.xticks(rotation=45, ha='right')
plt.title("Distribution of Forecastability Metrics")
plt.tight_layout()

In [ ]:
# plot more and less forecastable series
fig, ax = plt.subplots(len(use_cols), 2, figsize=(12, 30))
for i, m in enumerate(use_cols):
    idx_sorted = df_metrics[m].sort_values().index
    best_idxs = idx_sorted[-4:]
    worst_idxs = idx_sorted[:4]
    for j in range(3):
        best_s = series_list[best_idxs[j]]
        worst_s = series_list[worst_idxs[j]]
        ax[i, 0].plot(best_s / best_s.mean(), linewidth=1)
        ax[i, 1].plot(worst_s / worst_s.mean(), linewidth=1)
    # set titles
    ax[i, 0].set_title(f"Best 3: {m}")
    ax[i, 1].set_title(f"Worst 3: {m}")
plt.tight_layout()

# Visualization via UMAP
In the following cell we'll use UMAP to visualize the forecastability space defined by the indicators we computed.
UMAP (Uniform Manifold Approximation and Projection) is a dimensionality reduction technique based on manifold learning. As PCA, it can be used to project a multidimensional space into a lower dimensional one (usually 2D/3D for visualization).


The target (or y) option in UMAP's fit_transform method allows you to provide additional information (such as labels or a continuous variable) to guide the embedding. When specified, UMAP can use this target information to influence the layout, potentially making the embedding more discriminative with respect to the target. This is especially useful for supervised or semi-supervised visualization tasks.

In [ ]:
# use UMAP to visualize forecastability space
from umap.umap_ import UMAP
reducer = UMAP(random_state=RANDOM_SEED)
X_emb = reducer.fit_transform(df_comp[use_cols].to_numpy(dtype=float), y=df_metrics['forecastability_index'].to_numpy(dtype=float))


In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except:
    pass
from IPython.display import display
import ipywidgets as widgets
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_emb = df_metrics.copy()
df_emb['umap_x'] = X_emb[:,0]
df_emb['umap_y'] = X_emb[:,1]
df_emb.head()

id_to_series = {sid: s for sid, s in zip(series_ids, series_list)}
if 'series_id' not in df_emb.columns:
    raise KeyError("df_metrics must contain a 'series_id' column.")
default_color = 'forecastability_index' if 'forecastability_index' in df_emb.columns else use_cols[0]
fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42],
                    subplot_titles=('UMAP of Forecastability Metrics', 'Time Series Preview'))
scatter = go.Scattergl(
    x=df_emb['umap_x'],
    y=df_emb['umap_y'],
    mode='markers',
    marker=dict(size=6, color=df_emb[default_color], colorscale='Viridis', showscale=True,
                colorbar=dict(title=default_color)),
    text=df_emb['series_id'],
    hovertemplate='<b>%{text}</b><br>UMAP1=%{x:.3f}<br>UMAP2=%{y:.3f}<extra></extra>',
    name='UMAP points'
)
fig.add_trace(scatter, row=1, col=1)
line = go.Scatter(x=[], y=[], mode='lines', name='series')
fig.add_trace(line, row=1, col=2)
fig.update_xaxes(title_text='UMAP 1', row=1, col=1)
fig.update_yaxes(title_text='UMAP 2', row=1, col=1)
fig.update_xaxes(title_text='t (index)', row=1, col=2)
fig.update_yaxes(title_text='value (z-norm)', row=1, col=2)
fig.update_layout(height=600, margin=dict(l=10, r=10, t=40, b=10))
figw = go.FigureWidget(fig)
scatter_trace = figw.data[0]
line_trace = figw.data[1]
def znorm(x):
    x = np.asarray(x, float)
    if x.size == 0: return x
    m = np.nanmean(x); s = np.nanstd(x)
    return (x - m) / s if s > 0 else x - m
def hover_fn(trace, points, state):
    if not points.point_inds: return
    idx = points.point_inds[0]
    sid = df_emb.iloc[idx]['series_id']
    series = id_to_series.get(sid, None)
    if series is None or len(series) == 0:
        with figw.batch_update():
            line_trace.x = []; line_trace.y = []
            figw.layout.annotations[1].text = f'Time Series Preview (No data for {sid})'
        return
    x = np.arange(len(series)); y = znorm(series)
    with figw.batch_update():
        line_trace.x = x; line_trace.y = y; line_trace.name = str(sid)
        figw.layout.annotations[1].text = f'Time Series Preview — {sid}'
scatter_trace.on_hover(hover_fn)
options = [c for c in df_emb.columns if c not in ['umap_x','umap_y'] and df_emb[c].dtype != 'O']
if default_color not in options: options.insert(0, default_color)
color_dd = widgets.Dropdown(options=options, value=default_color, description='Color by:', layout=widgets.Layout(width='350px'))
def on_color_change(change):
    col = change['new']
    with figw.batch_update():
        figw.data[0].marker.color = df_emb[col]
        figw.data[0].marker.colorbar.title = col
color_dd.observe(on_color_change, names='value')
display(color_dd)
display(figw)

## 6) Leakage-safe multi-dataset transformation selection

This experiment asks whether the existing forecastability indices, plus `skewness`, `zero_share`, and `adf_logp`, help select a transformation that improves forecasts. Every diagnostic is computed only from the training prefix before the evaluation window.

We compare no transformation, Yeo--Johnson, first differencing, and seasonal differencing. Performance is measured on the original scale using MASE. The key external-validity test is leave-one-dataset-out: train on three domains and evaluate on the fourth.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import PowerTransformer

TEST_ORIGINS = 16
FORECAST_HORIZON = 6  # valid for each seasonal period, including daily NN5
LAGS = np.array([1, 2, 3, 4, 6, 12])
METHODS = ["none", "yeo_johnson", "diff1", "seasonal_diff"]

BASE_INDEXES = [
    "variance_ratio", "spectral_entropy", "spectral_forecastability",
    "periodicity_index", "acf1", "acf_mean_abs", "svd_entropy",
    "perm_entropy", "sample_entropy", "approx_entropy", "lzc",
    "ordpy_stat_complexity", "dfa_alpha",
]
FEATURE_COLUMNS = [c for c in BASE_INDEXES if c in df_metrics.columns] + [
    "skewness", "zero_share", "adf_logp",
]

experiment_panel = list(zip(series_datasets, series_ids, series_list, series_seasons))
print(f"Experiment: {len(experiment_panel)} series from {len(DATASET_SPECS)} datasets")
print("Features:", FEATURE_COLUMNS)


### ❓ Exercise 1 — direct forecasts after differencing

Complete the `diff1` and `seasonal_diff` branches in `design_for_method`.

For each branch define the transformed lag series `z`, the first valid forecast origin `first`, the horizon-ahead target in transformed space, the lagged feature extractor, and the inverse map back to the original scale.

The important part is alignment: the feature row at origin `o` must use only observations available before `o`, while the target may refer to the horizon-ahead value being predicted.

In [ ]:
def design_for_method(y, method, season, horizon=FORECAST_HORIZON):
    y = np.asarray(y, float)
    max_lag = int(np.max(LAGS))
    if method == "none":
        z = y; first = max_lag
        target = lambda o: y[o + horizon - 1]
        feature = lambda o, lag: z[o - lag]
        invert = lambda value: value
    elif method == "yeo_johnson":
        transformer = PowerTransformer(method="yeo-johnson", standardize=True)
        z = transformer.fit_transform(y.reshape(-1, 1)).ravel(); first = max_lag
        target = lambda o: z[o + horizon - 1]
        feature = lambda o, lag: z[o - lag]
        invert = lambda value: transformer.inverse_transform([[value]]).item()
    elif method == "diff1":
        # ? COMPLETE THIS BLOCK
        z = None
        first = None
        target = None
        feature = None
        invert = None
    elif method == "seasonal_diff":
        # ? COMPLETE THIS BLOCK
        z = None
        first = None
        target = None
        feature = None
        invert = None
    else:
        raise ValueError(method)
    X, target_y = [], []
    for origin in range(first, len(y) - horizon + 1):
        X.append([feature(origin, lag) for lag in LAGS])
        target_y.append(target(origin))
    next_row = np.asarray([feature(len(y), lag) for lag in LAGS], float)
    return np.asarray(X, float), np.asarray(target_y, float), next_row, invert

def direct_forecast(y_train, method, season):
    X, target, next_row, invert = design_for_method(y_train, method, season)
    if len(target) < 30:
        return np.nan
    mu, sd = X.mean(0), np.where(X.std(0) == 0, 1.0, X.std(0))
    Z = (X - mu) / sd
    target_mean = target.mean()  # unpenalised intercept
    coef = np.linalg.solve(Z.T @ Z + np.eye(Z.shape[1]), Z.T @ (target - target_mean))
    return float(invert(target_mean + ((next_row - mu) / sd) @ coef))

def mase_for_method(y, method, season):
    first_origin = len(y) - TEST_ORIGINS - FORECAST_HORIZON + 1
    errors = []
    for origin in range(first_origin, len(y) - FORECAST_HORIZON + 1):
        pred = direct_forecast(y[:origin], method, season)
        if np.isfinite(pred):
            errors.append(abs(y[origin + FORECAST_HORIZON - 1] - pred))
    scale = np.mean(np.abs(np.diff(y[:first_origin])))
    return np.mean(errors) / max(scale, 1e-12)


In [ ]:
records = []
for dataset, sid, y, season in experiment_panel:
    cutoff = len(y) - TEST_ORIGINS - FORECAST_HORIZON + 1
    metrics = compute_metrics_for_series(sid, y[:cutoff])
    scores = {method: mase_for_method(y, method, season) for method in METHODS}
    baseline = scores["none"]
    for method, mase in scores.items():
        records.append({
            "dataset": dataset, "series_id": sid, "seasonal_period": season,
            "method": method, "mase": mase,
            "relative_gain": (baseline - mase) / max(baseline, 1e-12),
            **{feature: metrics.get(feature, np.nan) for feature in FEATURE_COLUMNS},
        })

res = (pd.DataFrame(records).replace([np.inf, -np.inf], np.nan)
       .dropna(subset=["mase", "relative_gain"]))
display(res.groupby(["dataset", "method"])["mase"].agg(["median", "mean", "count"]).round(3))


### ❓ Exercise 2 — learn a leakage-safe transformation policy

Complete `score_policy`.

The function receives candidate transformation rows for train and test splits. Fit the gain model only on the training rows, predict `relative_gain` for each test candidate, select the candidate with the largest predicted gain for each series, and choose `none` whenever the best predicted gain is not positive.

The returned table should compare the learned policy with `always_best_train`, `none`, and the infeasible `oracle`.

In [ ]:
def make_policy_features(frame, columns=None):
    X = pd.get_dummies(frame[FEATURE_COLUMNS + ["method"]],
                       columns=["method"], dtype=float)
    return X if columns is None else X.reindex(columns=columns, fill_value=0.0)

def score_policy(train, test):
    X_train = make_policy_features(train)
    X_test = make_policy_features(test, X_train.columns)
    imputer = SimpleImputer(strategy="median")
    model = HistGradientBoostingRegressor(max_iter=160, max_leaf_nodes=10,
        learning_rate=0.06, l2_regularization=2.0, random_state=RANDOM_SEED)

    # ? Fit the model on training candidate rows and score test candidate rows.
    # ? COMPLETE THE NEXT TWO LINES
    scored = test.copy()
    scored["predicted_gain"] = None

    # ? Keep one selected candidate per series and turn negative predicted gains into no-transform decisions.
    selected = None
    selected["learned"] = None

    # ? Build realised-gain baselines for evaluation.
    actual = None
    default = None
    return pd.DataFrame({
        "learned": selected.set_index("series_id").learned.reindex(actual.index).fillna(0),
        "always_best_train": actual[default],
        "none": 0.0,
        "oracle": actual.max(axis=1).clip(lower=0),
    }), default

def summarise(frame):
    return frame.agg(["mean", "median", lambda x: (x > 0).mean()]).T.rename(
        columns={"<lambda>": "positive_rate"}
    )

cv = []
for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(res, groups=res.series_id)):
    result, _ = score_policy(res.iloc[tr], res.iloc[te])
    result["fold"] = fold
    cv.append(result)
cv_results = pd.concat(cv)
display(summarise(cv_results.drop(columns="fold")).round(3))

for held_out in sorted(res.dataset.unique()):
    result, default = score_policy(res[res.dataset != held_out], res[res.dataset == held_out])
    print(f"Held out {held_out} | training default: {default}")
    display(summarise(result).round(3))


### Interpretation

Pooled CV asks whether the diagnostics help when the training folds represent the same mixture of domains. Leave-one-dataset-out is the stronger result: a consistently positive `learned` value there would support a domain-robust transformation policy. Negative results for a held-out domain indicate that the policy is not yet safely transferable to that type of time series.

In [ ]:
# Interactive diagnostic-versus-gain view, using the same FigureWidget pattern as the UMAP cell above.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

from IPython.display import display
import ipywidgets as widgets
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# `relative_gain` = (MASE_none - MASE_transformation) / MASE_none.
plot_df = res.copy().replace([np.inf, -np.inf], np.nan)
plot_df = plot_df.dropna(subset=["dataset", "series_id", "method", "relative_gain"])
index_options = [c for c in FEATURE_COLUMNS if c in plot_df.columns]

series_lookup = {
    (dataset, sid): (np.asarray(y, dtype=float), season)
    for dataset, sid, y, season in experiment_panel
}

def znorm(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)
    return x - np.nanmean(x) if (not np.isfinite(sd) or sd == 0) else (x - np.nanmean(x)) / sd

def transform_for_preview(y, method, season):
    y = np.asarray(y, dtype=float)
    if method == "none":
        return np.arange(len(y)), y
    if method == "yeo_johnson":
        try:
            z = PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(y.reshape(-1, 1)).ravel()
            return np.arange(len(z)), z
        except Exception:
            return np.arange(len(y)), y
    if method == "diff1":
        return np.arange(1, len(y)), np.diff(y)
    if method == "seasonal_diff":
        lag = int(season) if pd.notna(season) and int(season) > 1 else 1
        return (np.arange(lag, len(y)), y[lag:] - y[:-lag]) if len(y) > lag else (np.arange(len(y)), y)
    return np.arange(len(y)), y

dataset_dd = widgets.Dropdown(options=["All datasets"] + sorted(plot_df.dataset.unique()), description="Dataset:", layout=widgets.Layout(width="300px"))
method_dd = widgets.Dropdown(options=sorted(plot_df.method.unique()), description="Transform:", layout=widgets.Layout(width="300px"))
index_dd = widgets.Dropdown(options=index_options, value=index_options[0], description="Index:", layout=widgets.Layout(width="300px"))

def selected_data():
    d = plot_df.loc[plot_df.method.eq(method_dd.value)].copy()
    if dataset_dd.value != "All datasets":
        d = d.loc[d.dataset.eq(dataset_dd.value)].copy()
    return d.dropna(subset=[index_dd.value, "relative_gain"]).sort_values(index_dd.value).reset_index(drop=True)

# Populate the first figure BEFORE converting it to a FigureWidget.  This is the
# same construction order used in the working UMAP visualization above.
current = {"data": selected_data()}
data = current["data"]
fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42],
                    subplot_titles=("Forecastability index versus realised gain", "Time-series preview"))
scatter = go.Scattergl(
    x=data[index_dd.value], y=data.relative_gain, mode="markers",
    marker=dict(size=7, color=data.relative_gain, colorscale="RdYlGn", cmid=0, showscale=True,
                colorbar=dict(title="gain")),
    text=[f"{r.dataset} — {r.series_id}" for r in data[["dataset", "series_id"]].itertuples(index=False)],
    hovertemplate="<b>%{text}</b><br>index=%{x:.4g}<br>relative gain=%{y:.2%}<extra></extra>",
    name="series",
)
fig.add_trace(scatter, row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#64748b", width=2), name="original"), row=1, col=2)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#ef4444", width=2), name="transformed"), row=1, col=2)
fig.update_xaxes(title_text=index_dd.value, row=1, col=1)
fig.update_yaxes(title_text="Relative MASE gain", tickformat=".0%", row=1, col=1)
fig.update_xaxes(title_text="t", row=1, col=2)
fig.update_yaxes(title_text="z-normalised value", row=1, col=2)
fig.update_layout(height=600, margin=dict(l=10, r=10, t=40, b=10))

figw = go.FigureWidget(fig)
scatter_trace = figw.data[0]
original_trace = figw.data[1]
transformed_trace = figw.data[2]

def refresh_plot(change=None):
    data = selected_data()
    current["data"] = data
    with figw.batch_update():
        scatter_trace.x = data[index_dd.value]
        scatter_trace.y = data.relative_gain
        scatter_trace.text = [f"{r.dataset} — {r.series_id}" for r in data[["dataset", "series_id"]].itertuples(index=False)]
        scatter_trace.marker.color = data.relative_gain
        figw.layout.xaxis.title = index_dd.value
        original_trace.x, original_trace.y = [], []
        transformed_trace.x, transformed_trace.y = [], []

def hover_fn(trace, points, state):
    if not points.point_inds:
        return
    row = current["data"].iloc[points.point_inds[0]]
    y, season = series_lookup[(row.dataset, row.series_id)]
    tx, ty = transform_for_preview(y, row.method, season)
    with figw.batch_update():
        original_trace.x, original_trace.y = np.arange(len(y)), znorm(y)
        transformed_trace.x, transformed_trace.y = tx, znorm(ty)
        original_trace.name = f"original — {row.series_id}"
        transformed_trace.name = row.method

scatter_trace.on_hover(hover_fn)
dataset_dd.observe(refresh_plot, names="value")
method_dd.observe(refresh_plot, names="value")
index_dd.observe(refresh_plot, names="value")

display(widgets.HBox([dataset_dd, method_dd, index_dd]))
display(figw)
